# Module 4 — Class 1: Linear Regression Challenge

**Level 3 — Exercises Only / Advanced Students**

You get minimal scaffolding. Build the full workflow yourself.

## Rules
- Use Superstore.
- Target: `Sales`.
- Use only features that would be available at prediction time unless you explicitly explain leakage risk.
- Keep train/test separation clean.
- Produce a final comparison table and written conclusions.

**Flow update:** live lesson now starts with a single Colab preparation block: **load → inspect → train/test split**. Homework/exercise logic remains the same, but use this order when explaining your notebook.


## Exercise 0 — Setup and Robust Loading

Use or modify this loader. Then proceed independently.

In [ ]:
# Robust Superstore dataset loader
# Loading modes:
# 1) preferred: local/pre-uploaded files in Colab
# 2) manual_upload: upload CSV manually
# 3) external_source: public GitHub fallback, then optional KaggleHub fallback

import os
import glob
import pandas as pd
import numpy as np

LOADING_MODE = "preferred"  # change to: "manual_upload" or "external_source"

COMMON_FILENAMES = [
    "superstore_cleaned.csv",
    "SampleSuperstore.csv",
    "Sample - Superstore.csv",
    "Superstore.csv",
    "superstore.csv",
]

COMMON_ENCODINGS = ["utf-8", "latin1", "windows-1252", "ISO-8859-1"]

def read_csv_safely(path_or_url):
    """Try common encodings and return a DataFrame."""
    last_error = None
    for enc in COMMON_ENCODINGS:
        try:
            return pd.read_csv(path_or_url, encoding=enc)
        except Exception as e:
            last_error = e
    raise last_error

def find_local_superstore_file():
    """Find a likely Superstore CSV in the current Colab folder."""
    for name in COMMON_FILENAMES:
        if os.path.exists(name):
            return name
    csv_files = glob.glob("*.csv")
    for file in csv_files:
        lower = file.lower()
        if "superstore" in lower or "store" in lower or "sales" in lower:
            return file
    return None

def load_superstore(mode="preferred"):
    """Load Superstore data using preferred, manual_upload, or external_source mode."""
    if mode == "preferred":
        local_file = find_local_superstore_file()
        if local_file:
            print(f"✅ Found local file: {local_file}")
            return read_csv_safely(local_file)
        print("⚠️ No local Superstore CSV found. Trying external source...")
        mode = "external_source"

    if mode == "manual_upload":
        try:
            from google.colab import files
            uploaded = files.upload()
            if not uploaded:
                raise ValueError("No file was uploaded.")
            file_name = list(uploaded.keys())[0]
            print(f"✅ Uploaded file: {file_name}")
            return read_csv_safely(file_name)
        except Exception as e:
            raise RuntimeError(
                "Manual upload failed. Please upload a Superstore CSV file, "
                "for example 'Sample - Superstore.csv'. Original error: " + str(e)
            )

    if mode == "external_source":
        urls = [
            "https://raw.githubusercontent.com/datasciencedojo/datasets/master/Superstore.csv",
        ]
        for url in urls:
            try:
                print(f"🌐 Trying public source: {url}")
                return read_csv_safely(url)
            except Exception as e:
                print("Could not load from this URL:", e)
        try:
            import kagglehub
            path = kagglehub.dataset_download("vivek468/superstore-dataset-final")
            csv_candidates = glob.glob(os.path.join(path, "*.csv"))
            if not csv_candidates:
                raise FileNotFoundError("KaggleHub downloaded the dataset, but no CSV file was found.")
            print(f"✅ KaggleHub file: {csv_candidates[0]}")
            return read_csv_safely(csv_candidates[0])
        except Exception as e:
            raise RuntimeError(
                "Could not load Superstore from local files, manual upload, or external sources.\n"
                "Try uploading 'Sample - Superstore.csv' manually.\n"
                "Original error: " + str(e)
            )

    raise ValueError("LOADING_MODE must be 'preferred', 'manual_upload', or 'external_source'.")

# Load data
df = load_superstore(LOADING_MODE)

# Normalize common column names lightly: strip spaces only, do not rename business columns.
df.columns = [str(c).strip() for c in df.columns]

print("✅ Data loaded!")
print("Shape:", df.shape)
print("Columns:", list(df.columns))

def find_column(possible_names):
    lower_map = {c.lower().replace(" ", "").replace("_", ""): c for c in df.columns}
    for name in possible_names:
        key = name.lower().replace(" ", "").replace("_", "")
        if key in lower_map:
            return lower_map[key]
    return None

TARGET_COL = find_column(["Sales"])
PROFIT_COL = find_column(["Profit"])
QUANTITY_COL = find_column(["Quantity"])
DISCOUNT_COL = find_column(["Discount"])

KEY_NUMERIC_COLS = [c for c in [TARGET_COL, QUANTITY_COL, DISCOUNT_COL, PROFIT_COL] if c is not None]

if TARGET_COL is None:
    raise ValueError("❌ Could not find the target column 'Sales'. Please check your dataset columns.")

print("🎯 Target column:", TARGET_COL)
print("🔢 Key numeric columns found:", KEY_NUMERIC_COLS)

df.head()

In [ ]:
# Add all imports you need here.

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, SGDRegressor  # Ridge/Lasso are optional later-module extensions
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Ready.")

## Exercise 1 — Data Audit

Produce a compact audit:
- shape;
- dtypes;
- missing values;
- numeric distributions;
- correlation matrix;
- at least one risk/limitation of the dataset for forecasting Sales.

In [ ]:
# EXERCISE 1 CODE HERE

## Exercise 2 — Baseline and Feature Set

Build at least two baselines:
1. naive mean predictor;
2. simple Linear Regression with one feature.

Then define a stronger feature set.

Optional: exclude `Profit` and explain why it may leak post-sale information.

In [ ]:
# EXERCISE 2 CODE HERE

## Exercise 3 — Multiple Linear Regression

Train and evaluate a multiple Linear Regression model.

Required metrics:
- MSE;
- RMSE;
- MAE;
- R².

Add coefficient interpretation.

In [ ]:
# EXERCISE 3 CODE HERE

## Exercise 4 — Polynomial Features with Pipeline

Train degree 2 and degree 3 polynomial models using `Pipeline`.

Compare train vs test metrics to detect overfitting.

In [ ]:
# EXERCISE 4 CODE HERE

## Exercise 5 — Residual Diagnostics

Create:
- predicted vs actual plot;
- residual vs predicted plot;
- residual histogram.

Write a diagnostic conclusion: underfitting, overfitting, non-linearity, outliers, or weak features?

In [ ]:
# EXERCISE 5 CODE HERE

## Exercise 6 — Gradient Descent / SGD Extension

Use `SGDRegressor` with scaling.

Compare it to closed-form `LinearRegression` / **OLS**.

Try at least two learning rates and explain the result.

**Do not add Ridge/Lasso here unless you clearly mark it as optional extension.** Regularization is covered later in Module 4.

In [ ]:
# EXERCISE 6 CODE HERE

## Exercise 7 — Final Model Selection Report

Create one final table:

| Model | Features | Train RMSE | Test RMSE | Test MAE | Test R² | Notes |
|---|---|---:|---:|---:|---:|---|

Then write 5–8 sentences:
1. Best model and why.
2. Whether polynomial features helped.
3. What coefficients/residuals tell you.
4. Biggest limitation.
5. What feature you would collect next in a real business project.

**Expected output:** one comparison table, three plots, and a written conclusion. Do not submit only code.

In [ ]:
# EXERCISE 7 CODE HERE